# Building a Spam Filter with Naive Bayes

In this project, we build a spam filter for SMS messages using the multinomial Naive Bayes algorithm. To teach the computer how to classify messages, we use a dataset of 5,572 SMS messages that were already classified by humans. The dataset was put together by Tiago A. Almeida and José María Gómez Hidalgo and can be downloaded from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/sms+spam+collection).

**Goal:** write a program that classifies new messages as spam or ham (non-spam) with an accuracy greater than 80%.

## Exploring the Dataset

In [ ]:
import pandas as pd

sms_spam = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['Label', 'SMS'])

print(sms_spam.shape)
sms_spam.head()

In [ ]:
sms_spam['Label'].value_counts(normalize=True) * 100

The dataset has 5,572 rows and 2 columns. About **86.6%** of the messages are ham (non-spam) and the remaining **13.4%** are spam. This sample looks representative, since in practice most messages people receive are ham.

## Training and Test Set

Once the spam filter is built, we need to test how good it is at classifying new messages. We split the dataset into two categories:

- A **training set** (80% of the data) that we use to "train" the computer how to classify messages.
- A **test set** (20% of the data) that we use to test how good the spam filter is at classifying new messages.

We start by randomizing the dataset so that spam and ham messages are spread evenly across the two sets.

In [ ]:
data_randomized = sms_spam.sample(frac=1, random_state=1)

training_test_index = round(len(data_randomized) * 0.8)

training_set = data_randomized[:training_test_index].reset_index(drop=True)
test_set = data_randomized[training_test_index:].reset_index(drop=True)

print(training_set.shape)
print(test_set.shape)

In [ ]:
training_set['Label'].value_counts(normalize=True) * 100

In [ ]:
test_set['Label'].value_counts(normalize=True) * 100

The training set has 4,458 messages and the test set 1,114 messages. The percentages of spam and ham in both sets (86.5% / 13.5% for training, 86.8% / 13.2% for test) are very close to what we have in the full dataset, so both sets are representative.

## Data Cleaning

To calculate all the probabilities required by the algorithm, we first bring the training set to a format where each unique word in the vocabulary is a column and each row shows how many times that word occurs in a message.

### Letter Case and Punctuation

We start by removing all the punctuation and bringing every letter to lower case.

In [ ]:
# Before cleaning
training_set.head(3)

In [ ]:
# After cleaning
training_set['SMS'] = training_set['SMS'].str.replace(r'\W', ' ', regex=True)
training_set['SMS'] = training_set['SMS'].str.lower()
training_set.head(3)

### Creating the Vocabulary

Next, we create a list with all the unique words that occur in the messages of the training set.

In [ ]:
training_set['SMS'] = training_set['SMS'].str.split()

vocabulary = []
for sms in training_set['SMS']:
    for word in sms:
        vocabulary.append(word)

vocabulary = list(set(vocabulary))
len(vocabulary)

There are **7,783 unique words** across all the messages of the training set.

### The Final Training Set

We now use the vocabulary to count, for each word, the number of times it occurs in each message. Conceptually this is a dictionary where each key is a word and each value is a list of counts (one per message); we store the same numbers in a compact NumPy array to keep memory usage low. We then turn it into a DataFrame and concatenate it with the training set.

In [ ]:
import numpy as np

# Same idea as the dictionary {word: [count in sms 0, count in sms 1, ...]},
# but stored in a compact uint8 NumPy array to keep memory usage low
# (4,458 messages x 7,700+ words would take hundreds of MB as Python lists / int64)
word_index = {word: i for i, word in enumerate(vocabulary)}
counts = np.zeros((len(training_set['SMS']), len(vocabulary)), dtype=np.uint8)

for index, sms in enumerate(training_set['SMS']):
    for word in sms:
        counts[index, word_index[word]] += 1

word_counts = pd.DataFrame(counts, columns=vocabulary)
word_counts.head()

In [ ]:
training_set_clean = pd.concat([training_set, word_counts], axis=1)
training_set_clean.head()

## Calculating Constants First

The Naive Bayes algorithm answers these two probability questions to classify a new message:

$$P(Spam | w_1, w_2, ..., w_n) \propto P(Spam) \cdot \prod_{i=1}^{n}P(w_i|Spam)$$

$$P(Ham | w_1, w_2, ..., w_n) \propto P(Ham) \cdot \prod_{i=1}^{n}P(w_i|Ham)$$

To calculate $P(w_i|Spam)$ and $P(w_i|Ham)$ we use these equations:

$$P(w_i|Spam) = \frac{N_{w_i|Spam} + \alpha}{N_{Spam} + \alpha \cdot N_{Vocabulary}}$$

$$P(w_i|Ham) = \frac{N_{w_i|Ham} + \alpha}{N_{Ham} + \alpha \cdot N_{Vocabulary}}$$

Some of these terms have the same value for every new message: $P(Spam)$, $P(Ham)$, $N_{Spam}$, $N_{Ham}$ and $N_{Vocabulary}$. We use Laplace smoothing, so $\alpha = 1$.

In [ ]:
# Isolating spam and ham messages first
spam_messages = training_set_clean[training_set_clean['Label'] == 'spam']
ham_messages = training_set_clean[training_set_clean['Label'] == 'ham']

# P(Spam) and P(Ham)
p_spam = len(spam_messages) / len(training_set_clean)
p_ham = len(ham_messages) / len(training_set_clean)

# N_Spam
n_words_per_spam_message = spam_messages['SMS'].apply(len)
n_spam = n_words_per_spam_message.sum()

# N_Ham
n_words_per_ham_message = ham_messages['SMS'].apply(len)
n_ham = n_words_per_ham_message.sum()

# N_Vocabulary
n_vocabulary = len(vocabulary)

# Laplace smoothing
alpha = 1

print(p_spam, p_ham, n_spam, n_ham, n_vocabulary)

## Calculating Parameters

The parameters $P(w_i|Spam)$ and $P(w_i|Ham)$ depend only on the training set, so we calculate them once for every word in the vocabulary and store them in two dictionaries.

In [ ]:
# Initiate parameters
parameters_spam = {unique_word: 0 for unique_word in vocabulary}
parameters_ham = {unique_word: 0 for unique_word in vocabulary}

# Calculate parameters
for word in vocabulary:
    n_word_given_spam = spam_messages[word].sum()
    p_word_given_spam = (n_word_given_spam + alpha) / (n_spam + alpha * n_vocabulary)
    parameters_spam[word] = p_word_given_spam

    n_word_given_ham = ham_messages[word].sum()
    p_word_given_ham = (n_word_given_ham + alpha) / (n_ham + alpha * n_vocabulary)
    parameters_ham[word] = p_word_given_ham

## Classifying a New Message

With all the parameters calculated, we can write the spam filter. The `classify()` function:

- Takes in a new message as input.
- Cleans the message the same way we cleaned the training set.
- Calculates $P(Spam|w_1, w_2, ..., w_n)$ and $P(Ham|w_1, w_2, ..., w_n)$, ignoring words that aren't in the vocabulary.
- Compares the two values and classifies the message as spam or ham (or asks for human help if the values are equal).

In [ ]:
import re

def classify(message):
    message = re.sub(r'\W', ' ', message)
    message = message.lower().split()

    p_spam_given_message = p_spam
    p_ham_given_message = p_ham

    for word in message:
        if word in parameters_spam:
            p_spam_given_message *= parameters_spam[word]

        if word in parameters_ham:
            p_ham_given_message *= parameters_ham[word]

    print('P(Spam|message):', p_spam_given_message)
    print('P(Ham|message):', p_ham_given_message)

    if p_ham_given_message > p_spam_given_message:
        print('Label: Ham')
    elif p_ham_given_message < p_spam_given_message:
        print('Label: Spam')
    else:
        print('Equal probabilities, have a human classify this!')

In [ ]:
classify('WINNER!! This is the secret code to unlock the money: C3421.')

In [ ]:
classify('Sounds good, Tom, then see u there')

The two test messages were classified as expected: the first one (a typical "you won a prize" message) as spam, and the second one (a casual reply to a friend) as ham.

## Measuring the Spam Filter's Accuracy

We now test the filter on the 1,114 messages in the test set. The function below is the same as `classify()`, but it returns the label instead of printing it.

In [ ]:
def classify_test_set(message):
    message = re.sub(r'\W', ' ', message)
    message = message.lower().split()

    p_spam_given_message = p_spam
    p_ham_given_message = p_ham

    for word in message:
        if word in parameters_spam:
            p_spam_given_message *= parameters_spam[word]

        if word in parameters_ham:
            p_ham_given_message *= parameters_ham[word]

    if p_ham_given_message > p_spam_given_message:
        return 'ham'
    elif p_spam_given_message > p_ham_given_message:
        return 'spam'
    else:
        return 'needs human classification'

In [ ]:
test_set['predicted'] = test_set['SMS'].apply(classify_test_set)
test_set.head()

In [ ]:
correct = 0
total = test_set.shape[0]

for row in test_set.iterrows():
    row = row[1]
    if row['Label'] == row['predicted']:
        correct += 1

print('Correct:', correct)
print('Incorrect:', total - correct)
print('Accuracy:', correct / total)

The spam filter correctly classified **1,100 out of 1,114** messages in the test set, an accuracy of about **98.74%** — well above our 80% goal.

Only 14 messages were misclassified:

- 8 spam messages were labeled as ham,
- 5 ham messages were labeled as spam,
- 1 ham message got equal probabilities and was sent for human classification.

## Next Steps

We built a spam filter for SMS messages using the multinomial Naive Bayes algorithm. It reached an accuracy of 98.74% on the test set, much better than the 80% we initially aimed for.

Possible next steps:

- Analyze the 14 misclassified messages to understand why the algorithm got them wrong.
- Make the filtering process more complex by making the algorithm sensitive to letter case.
- Try removing very common words (stop words) or using different values of the smoothing parameter alpha.